# Fine-tune PaliGemma on RSVQA-LR (our own VQA checkpoint)

**Goal:** the mandatory single-image VQA baseline (`models/vqa/vqa_tool.py`,
`backend/app/specialists/vqa_adapter.py`) ships in Stage 1 with Google's own
`google/paligemma-3b-ft-rsvqa-lr-224` -- already fine-tuned by Google on this exact benchmark. This
notebook produces **our own** fine-tune, starting from the *pretrained* (not task-tuned) checkpoint
`google/paligemma-3b-pt-224` and LoRA-tuning it on RSVQA-LR ourselves, per the plan's Stage 2.

**Prompt format must match `vqa_tool.py` exactly.** The wrapper sends the bare question as the
prompt (no `"answer en"` prefix) -- see its `answer()` docstring. Training uses the same bare
question as the prefix and the ground-truth answer as the suffix, so a checkpoint trained here is
literally a drop-in: no wrapper code changes, just point `VQA_MODEL_ID` (or `vqa_tool.py`'s
`--checkpoint`) at the exported local directory.

**Data:** RSVQA-LR is fully local already (`data/raw/rsvqa/LR/`, 91MB images + JSON Q/A) -- this
notebook re-downloads the same public Zenodo record (`10.5281/zenodo.6344333`) directly on Kaggle
rather than requiring an upload, same as the DIOR-RSVG download in the grounding notebooks.

**Disk note (learned the hard way in the grounding notebooks):** the Hugging Face model cache lives
under `/root/.cache/huggingface`, *not* `/kaggle/working` -- so the ~11.7GB base-model download
does not count against the 20GB `/kaggle/working` output quota. Only the final exported checkpoint
(saved in bf16, ~6GB) lands in `/kaggle/working/output_model/`.

10-section structure, same as the other three training notebooks: setup -> HF auth -> data ->
dataset/collator -> base model + LoRA -> train -> eval (per-type accuracy, RSVQA's own metric,
zero-shot vs fine-tuned) -> export -> next steps.

In [ ]:
import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    print(" -", torch.cuda.get_device_name(i))

## 1. Setup

In [ ]:
!pip install -q -U transformers accelerate peft

## 2. Hugging Face auth

`google/paligemma-3b-pt-224` is gated behind the same Gemma license as the Stage-1 checkpoint.
Add your HF token as a **Kaggle secret** named `HF_TOKEN` (Add-ons -> Secrets in the notebook
editor) -- this keeps it out of the notebook file and any output. You need to have accepted the
license once at https://huggingface.co/google/paligemma-3b-pt-224 with the same account.

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
login(token=HF_TOKEN)
print("Logged in to the Hub.")

## 3. Download RSVQA-LR

Same Zenodo record `data/scripts/download_rsvqa.py` pulls from locally
(https://zenodo.org/records/6344333) -- 772 images (91MB zip) + question/answer/image-metadata
JSONs for the train/val/test splits. Downloaded straight into `/kaggle/working` since the whole
dataset is under 100MB, nowhere near the 20GB quota.

In [ ]:
import os, requests

DATA_DIR = "/kaggle/working/rsvqa_lr"
os.makedirs(DATA_DIR, exist_ok=True)

ZENODO_RECORD = "6344333"
files = requests.get(f"https://zenodo.org/api/records/{ZENODO_RECORD}").json()["files"]
NEEDED = {
    "Images_LR.zip",
    "LR_split_train_questions.json", "LR_split_train_answers.json", "LR_split_train_images.json",
    "LR_split_val_questions.json", "LR_split_val_answers.json", "LR_split_val_images.json",
    "LR_split_test_questions.json", "LR_split_test_answers.json", "LR_split_test_images.json",
}

for f in files:
    key = f["key"]
    if key not in NEEDED:
        continue
    dest = os.path.join(DATA_DIR, key)
    if os.path.exists(dest):
        print(f"[skip] {key} (already present)")
        continue
    print(f"Downloading {key} ...")
    r = requests.get(f["links"]["self"], stream=True)
    r.raise_for_status()
    with open(dest, "wb") as out:
        for chunk in r.iter_content(chunk_size=1 << 20):
            out.write(chunk)

import zipfile
with zipfile.ZipFile(os.path.join(DATA_DIR, "Images_LR.zip")) as z:
    z.extractall(DATA_DIR)

IMAGES_DIR = os.path.join(DATA_DIR, "Images_LR")
print("Images:", len(os.listdir(IMAGES_DIR)), "files in", IMAGES_DIR)

## 4. Build (image, question, answer) triples

Each split's questions/answers/images JSONs are joined by id: a question names its `img_id` and a
list of `answers_ids`; an answer names its `question_id`. RSVQA-LR images are named `<img_id>.tif`
inside the zip. Kept as flat triples per split -- no need to group by image for this task.

In [ ]:
import json

def load_split(name):
    q = json.load(open(os.path.join(DATA_DIR, f"LR_split_{name}_questions.json")))["questions"]
    a = json.load(open(os.path.join(DATA_DIR, f"LR_split_{name}_answers.json")))["answers"]
    answer_by_question_id = {ans["question_id"]: ans["answer"] for ans in a}

    triples = []
    for question in q:
        answer = answer_by_question_id.get(question["id"])
        if answer is None:
            continue
        image_path = os.path.join(IMAGES_DIR, f"{question['img_id']}.tif")
        if not os.path.exists(image_path):
            continue
        triples.append({
            "image_path": image_path,
            "question": question["question"],
            "answer": answer,
            "type": question["type"],
        })
    return triples

train_triples = load_split("train")
val_triples = load_split("val")
test_triples = load_split("test")
print(f"train={len(train_triples)}  val={len(val_triples)}  test={len(test_triples)}")
print("Example:", train_triples[0])

## 5. Dataset + collator

PaliGemma's own processor supports supervised fine-tuning directly: pass the question as `text`
(the prefix, i.e. the prompt) and the ground-truth answer as `suffix` -- the processor builds
`input_ids` as prefix+suffix and `labels` with the prefix (and image tokens) masked to `-100`, so
loss is only computed on the answer tokens. This is the documented PaliGemma fine-tuning pattern.

In [ ]:
from PIL import Image
from torch.utils.data import Dataset

class RSVQADataset(Dataset):
    def __init__(self, triples):
        self.triples = triples

    def __len__(self):
        return len(self.triples)

    def __getitem__(self, idx):
        t = self.triples[idx]
        image = Image.open(t["image_path"]).convert("RGB")
        return {"image": image, "question": t["question"], "answer": t["answer"]}


def make_collate_fn(processor):
    def collate_fn(batch):
        images = [b["image"] for b in batch]
        questions = [b["question"] for b in batch]
        answers = [b["answer"] for b in batch]
        inputs = processor(
            images=images,
            text=questions,
            suffix=answers,
            return_tensors="pt",
            padding="longest",
            tokenize_newline_separately=False,
        )
        return inputs
    return collate_fn

## 6. Base model + LoRA

Base weights load in bf16 (~6GB resident) with the vision tower and language-model backbone frozen
-- only LoRA adapters on the attention projections train, which keeps this well within a single
T4/P100's 16GB and makes the exported delta tiny relative to a full fine-tune.

In [ ]:
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration
from peft import LoraConfig, get_peft_model

BASE_MODEL_ID = "google/paligemma-3b-pt-224"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoProcessor.from_pretrained(BASE_MODEL_ID)
model = PaliGemmaForConditionalGeneration.from_pretrained(BASE_MODEL_ID, dtype=torch.bfloat16).to(DEVICE)

for param in model.vision_tower.parameters():
    param.requires_grad = False
for param in model.multi_modal_projector.parameters():
    param.requires_grad = False

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 7. Train

Plain PyTorch loop (consistent with the water-U-Net notebook) rather than `Trainer`, so the
training/eval split stays easy to follow. `EPOCHS=2` over ~77k train questions is already a lot of
gradient steps for a LoRA adapter at this rank -- watch val loss and stop earlier if it plateaus.

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 4
EPOCHS = 2
LR = 2e-4

train_loader = DataLoader(
    RSVQADataset(train_triples), batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=make_collate_fn(processor),
)
val_loader = DataLoader(
    RSVQADataset(val_triples), batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=make_collate_fn(processor),
)

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)

def run_eval_loss():
    model.eval()
    total, count = 0.0, 0
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            loss = model(**batch).loss
            total += loss.item()
            count += 1
    return total / max(count, 1)

for epoch in range(EPOCHS):
    model.train()
    running = 0.0
    for step, batch in enumerate(train_loader):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        running += loss.item()
        if step % 500 == 0:
            print(f"epoch {epoch} step {step}/{len(train_loader)} loss {loss.item():.4f}")
    val_loss = run_eval_loss()
    print(f"== epoch {epoch} done | train loss {running / len(train_loader):.4f} | val loss {val_loss:.4f} ==")

## 8. Evaluate: per-type accuracy on the held-out test split

RSVQA's own metric, so this is directly comparable to published baselines and to Stage 1's
off-the-shelf checkpoint. Greedy-decodes each test question the same way `vqa_tool.py` does
(bare question prompt, `max_new_tokens=20`) and compares the normalized answer string.

In [ ]:
from collections import defaultdict

def normalize(text):
    return text.strip().lower().rstrip(".")

def evaluate(model, triples, max_examples=None, batch_size=8):
    model.eval()
    subset = triples[:max_examples] if max_examples else triples
    correct_by_type = defaultdict(int)
    total_by_type = defaultdict(int)

    for i in range(0, len(subset), batch_size):
        batch = subset[i:i + batch_size]
        images = [Image.open(t["image_path"]).convert("RGB") for t in batch]
        questions = [t["question"] for t in batch]
        inputs = processor(images=images, text=questions, return_tensors="pt", padding="longest").to(DEVICE)
        input_len = inputs["input_ids"].shape[-1]
        with torch.no_grad():
            generated = model.generate(**inputs, max_new_tokens=20, do_sample=False)
        for t, gen in zip(batch, generated):
            predicted = normalize(processor.decode(gen[input_len:], skip_special_tokens=True))
            correct_by_type[t["type"]] += int(predicted == normalize(t["answer"]))
            total_by_type[t["type"]] += 1

    print(f"{'type':<15} {'accuracy':>10} {'n':>8}")
    overall_correct, overall_total = 0, 0
    for qtype in sorted(total_by_type):
        c, n = correct_by_type[qtype], total_by_type[qtype]
        print(f"{qtype:<15} {c / n:>10.3f} {n:>8}")
        overall_correct += c
        overall_total += n
    print(f"{'overall':<15} {overall_correct / overall_total:>10.3f} {overall_total:>8}")

# Cap at a few thousand for a reasonable eval time; raise/remove the cap for the final reported number.
EVAL_SUBSET = 3000
print("Fine-tuned:")
evaluate(model, test_triples, max_examples=EVAL_SUBSET)

Optional: compare against zero-shot (no fine-tuning) to see what the LoRA actually bought. Loads a
second copy of the *base* pretrained model with no adapter -- skip this cell if GPU memory is
tight, it's not required for the export step below.

In [ ]:
base_only = PaliGemmaForConditionalGeneration.from_pretrained(BASE_MODEL_ID, dtype=torch.bfloat16).to(DEVICE)
base_only.eval()
print("Zero-shot (no fine-tuning):")
evaluate(base_only, test_triples, max_examples=EVAL_SUBSET)
del base_only
torch.cuda.empty_cache()

## 9. Export

Merges the LoRA adapter into the base weights and saves a plain (non-PEFT) PaliGemma checkpoint in
bf16 -- so it loads through the *exact same* `PaliGemmaForConditionalGeneration.from_pretrained(...)`
call `vqa_tool.py` already makes, just pointed at a local directory instead of a Hub id. No wrapper
code changes. Saved in bf16 (not the Hub's usual fp32), so this download is ~6GB, not ~11.7GB.

In [ ]:
merged = model.merge_and_unload()

OUTPUT_DIR = "/kaggle/working/output_model/paligemma-rsvqa-lr-finetuned"
os.makedirs(OUTPUT_DIR, exist_ok=True)
merged.save_pretrained(OUTPUT_DIR, safe_serialization=True)
processor.save_pretrained(OUTPUT_DIR)
print("Exported to", OUTPUT_DIR)
!du -sh {OUTPUT_DIR}

## Next

Download the `output_model/paligemma-rsvqa-lr-finetuned/` folder from this notebook's Output tab
(zip it first from the Output panel, or download file-by-file) and place it locally at
`models/vqa/checkpoints/paligemma-rsvqa-lr-finetuned/`.

To use it, either pass the local path directly:

```bash
python models/vqa/vqa_tool.py --model-id models/vqa/checkpoints/paligemma-rsvqa-lr-finetuned --image <test image> --question "Is there a road in this image?"
```

or point the backend at it by setting in `.env`:

```
VQA_MODEL_ID=/Users/parixitsingh/Desktop/SATQUERY-AI/models/vqa/checkpoints/paligemma-rsvqa-lr-finetuned
```

(`AutoProcessor.from_pretrained` / `PaliGemmaForConditionalGeneration.from_pretrained` both accept
a local directory exactly like a Hub id -- `HF_TOKEN` is then irrelevant for this checkpoint since
nothing gated is being fetched.) Compare the per-type accuracy printed above against Stage 1's
`google/paligemma-3b-ft-rsvqa-lr-224` on the same test questions before deciding whether to switch
the default -- Google's checkpoint was already tuned on this exact benchmark, so beating it isn't
guaranteed with 2 epochs of rank-8 LoRA; this notebook exists to have *our own* trained checkpoint
on record either way, per the spec's expectation that models are fine-tuned, not just called.